# Data Exploration and Cleaning

This notebook focuses on understanding the TB outcome dataset, identifying data quality issues, handling missing values, removing duplicates, and standardizing inconsistencies to prepare the data for analysis.





In [4]:
# import necessary dependancies
import duckdb
import pandas as pd

In [10]:

data_path='../data/tb-outcomes.csv'


## Data Exploration

In [14]:
# display 5 first observation
duckdb.sql(f"""
           SELECT *
           FROM read_csv_auto('{data_path}')
           LIMIT 5
           """).fetchdf()

,patient_id,age,sex,region,diagnosis_date,hiv_status,diabetes,smear_result,drug_resistance,planned_months,weight_start_kg,weight_end_kg,adherence_pct,outcome,treatment_success
0,TB30000,44,Female,Capital,2022-07-15,Unknown,False,Negative,Drug-sensitive,6,55.5,59.3,90.9,Cured,1
1,TB30001,47,Male,South,2022-03-28,Negative,False,Not done,Drug-sensitive,6,51.7,53.3,73.5,Treatment completed,1
2,TB30002,19,Female,North,2021-10-08,HIV+,False,Negative,Drug-sensitive,6,66.7,73.7,74.6,Treatment completed,1
3,TB30003,20,Female,West,2023-05-10,Negative,False,Positive,Drug-sensitive,6,62.3,58.6,96.5,Died,0
4,TB30004,37,Male,South,2022-06-14,+,False,Positive,Drug-sensitive,6,48.9,54.1,79.9,Cured,1


In [ ]:
# count number of rows
duckdb.sql(f"""
           SELECT COUNT(*) as number_rows
           FROM  read_csv_auto('{data_path}')
           """).fetchdf().loc[0,'number_rows']

np.int64(3000)

In [25]:
# get general info of data
duckdb.sql(f"""
           DESCRIBE
           SELECT *
           FROM read_csv_auto('{data_path}')
           """).fetchdf()

,column_name,column_type,null,key,default,extra
0,patient_id,VARCHAR,YES,None,None,None
1,age,BIGINT,YES,None,None,None
2,sex,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,diagnosis_date,VARCHAR,YES,None,None,None
5,hiv_status,VARCHAR,YES,None,None,None
6,diabetes,BOOLEAN,YES,None,None,None
7,smear_result,VARCHAR,YES,None,None,None
8,drug_resistance,VARCHAR,YES,None,None,None
9,planned_months,BIGINT,YES,None,None,None


There are 3000 rows and 14 columns

In [67]:
# list all table columns
columns=duckdb.sql(
    f""" 
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{data_path}')
    """
).fetchdf()["column_name"].tolist()

In [ ]:
# calculate some of missing values per column
column_null_count={}
for col in columns:
    count=duckdb.sql(
        f""" 
        SELECT count('{col}') as {col}
        FROM read_csv_auto('{data_path}')
        WHERE {col} IS NULL
        """
    ).fetchdf()
  
    column_null_count[col]=int(count[col][0])
    
column_null_count

{'patient_id': 0,
 'age': 0,
 'sex': 0,
 'region': 0,
 'diagnosis_date': 0,
 'hiv_status': 0,
 'diabetes': 0,
 'smear_result': 0,
 'drug_resistance': 0,
 'planned_months': 0,
 'weight_start_kg': 0,
 'weight_end_kg': 300,
 'adherence_pct': 150,
 'outcome': 0,
 'treatment_success': 0}

In [72]:
# check for duplicates
duckdb.sql(
    f""" 
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT patient_id
        FROM read_csv_auto('{data_path}'))
    """
).fetchdf()

,count_star()
0,3000


There are no duplicate since they are 3000 rows and 3000 distinct rows

In [ ]:
#  unique values per column
unique_values=[]
for col in columns:
    result=duckdb.sql(f"""
               SELECT DISTINCT {col}
               FROM read_csv_auto('{data_path}')
               """).fetchdf()
    unique_values.append(result)
unique_values

[     patient_id
 0       TB30023
 1       TB30043
 2       TB30059
 3       TB30116
 4       TB30123
 ...         ...
 2995    TB32939
 2996    TB32943
 2997    TB32950
 2998    TB32951
 2999    TB32956
 
 [3000 rows x 1 columns],
     age
 0    70
 1    42
 2    53
 3    26
 4    11
 ..  ...
 79   58
 80   41
 81   76
 82    6
 83   82
 
 [84 rows x 1 columns],
       sex
 0  Female
 1    Male,
     region
 0  Capital
 1    South
 2     West
 3    North
 4     East,
      diagnosis_date
 0        08/09/2023
 1        29/11/2022
 2        2021-10-26
 3        28/03/2021
 4        2021-04-13
 ...             ...
 1769     2022-10-26
 1770   Jan 16, 2022
 1771     2023-04-11
 1772     05/02/2021
 1773     25/03/2021
 
 [1774 rows x 1 columns],
     hiv_status
 0            -
 1          neg
 2            +
 3            ?
 4   Not tested
 5      Unknown
 6         HIV-
 7     Positive
 8          pos
 9     Negative
 10        HIV+,
    diabetes
 0     False
 1      True,
   smear_resul

In [82]:
# inspect rows with missing values
duckdb.sql(f"""
           SELECT *
           FROM read_csv_auto('{data_path}')
           WHERE weight_end_kg IS NULL
           OR adherence_pct IS NULL
;
           """)

patient_idVARCHAR,ageBIGINT,sexVARCHAR,regionVARCHAR,diagnosis_dateVARCHAR,hiv_statusVARCHAR,diabetesBOOLEAN,smear_resultVARCHAR,drug_resistanceVARCHAR,planned_monthsBIGINT,weight_start_kgDOUBLE,weight_end_kgDOUBLE,adherence_pctDOUBLE,outcomeVARCHAR,treatment_successBIGINT
TB30008,21,Male,North,01/03/2022,Negative,False,Positive,Drug-sensitive,6,42.3,47.5,NULL,Treatment completed,1
TB30021,44,Female,North,"May 12, 2023",Negative,False,Positive,Drug-sensitive,6,46.4,NULL,74.0,Cured,1
TB30025,43,Male,Capital,2023-11-07,Negative,False,Positive,Drug-sensitive,6,55.2,NULL,92.0,Cured,1
TB30032,52,Male,Capital,"Oct 23, 2021",neg,False,Negative,Drug-sensitive,6,49.5,NULL,80.5,Cured,1
TB30033,40,Male,Capital,28/09/2021,Negative,True,Negative,Drug-sensitive,6,68.5,73.3,NULL,Treatment completed,1
TB30038,70,Male,Capital,2021-01-16,Positive,False,Positive,Drug-sensitive,6,61.8,NULL,92.9,Cured,1
TB30057,50,Male,South,2022-01-02,Negative,False,Not done,Drug-sensitive,6,53.8,NULL,89.9,Treatment completed,1
TB30058,42,Male,South,30/11/2021,Negative,False,Positive,Drug-sensitive,6,38.8,NULL,82.6,Treatment completed,1
TB30062,9,Male,West,2021-11-17,Negative,False,Positive,Drug-sensitive,6,43.0,43.4,NULL,Cured,1
TB30081,30,Female,North,29/09/2021,Negative,False,Positive,Drug-sensitive,6,55.5,NULL,90.8,Cured,1


## Data Cleaning Tasks

The following data quality issues were identified during data exploration and will be addressed during cleaning:

1. **Standardize `hiv_status`:** The column contains inconsistent values representing HIV status, including `neg`, `Negative`, `HIV-`, `pos`, `Positive`, `HIV+`, `Not tested`, `Unknown`, and `?`, . Standardize these values into consistent categories while preserving unknown and untested statuses.

2. **Convert `diagnosis_date`:** Convert the `diagnosis_date` column from its current data type to a proper date format.

3. **Handle missing values:** Investigate and address missing values in `weight_end_kg` and `adherence_pct`.

4. **Standardize drug resistance values:** Convert the drug resistance categories (`MDR` and `Drug Sensitive`) into Boolean values (`True` and `False`), respectively.

### Data Cleaning

In [87]:
def wrangle_tb_outcome_data(csv_path):
    duckdb.sql(f"""
               CREATE OR REPLACE TABLE tb AS
               SELECT *
               FROM read_csv_auto('{csv_path}')
               """)
    duckdb.sql(
        """ 
        UPDATE tb
        SET hiv_status = CASE
            WHEN LOWER(TRIM(hiv_status)) IN ('neg','negative','hiv-','-')
                THEN 'Negative' 
            WHEN LOWER(TRIM(hiv_status)) IN ('pos','positive','hiv+','+')
                THEN 'Positive'
            WHEN LOWER(TRIM(hiv_status)) IN ('?','not tested','unknown')
                THEN 'Unknown'
            ELSE 'Unknown'
            END
        """
        
    )
    duckdb.sql(
        """ 
        UPDATE tb
        SET drug_resistance = CASE
        WHEN LOWER(TRIM(drug_resistance))='Drug-sensitive'
            THEN True
        ELSE  False
        END
        
        """
    )
    
    duckdb.sql(
        """ 
        ALTER TABLE tb
        ALTER COLUMN diagnosis_date SET DATA TYPE DATE
        USING TRY_CAST(diagnosis_date AS DATE)
        
        """
    )
    
    return duckdb.sql("SELECT * FROM tb").fetchdf()

    
    

In [88]:
wrangle_tb_outcome_data(data_path)

,patient_id,age,sex,region,diagnosis_date,hiv_status,diabetes,smear_result,drug_resistance,planned_months,weight_start_kg,weight_end_kg,adherence_pct,outcome,treatment_success
0,TB30000,44,Female,Capital,2022-07-15,Unknown,False,Negative,false,6,55.5,59.3,90.9,Cured,1
1,TB30001,47,Male,South,2022-03-28,Negative,False,Not done,false,6,51.7,53.3,73.5,Treatment completed,1
2,TB30002,19,Female,North,2021-10-08,Positive,False,Negative,false,6,66.7,73.7,74.6,Treatment completed,1
3,TB30003,20,Female,West,2023-05-10,Negative,False,Positive,false,6,62.3,58.6,96.5,Died,0
4,TB30004,37,Male,South,2022-06-14,Positive,False,Positive,false,6,48.9,54.1,79.9,Cured,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,TB32995,37,Female,North,NaT,Negative,True,Negative,false,6,47.8,50.0,44.8,Treatment completed,1
2996,TB32996,36,Male,South,NaT,Unknown,False,Positive,false,6,52.3,54.8,90.3,Treatment completed,1
2997,TB32997,30,Male,North,2022-07-16,Negative,True,Negative,false,6,63.8,71.0,78.6,Cured,1
2998,TB32998,59,Female,East,NaT,Negative,False,Not done,false,6,64.6,68.0,86.5,Treatment completed,1
